# run_gemini_test.py on Kaggle (test-set per-card Gemini transcription)

Runs `src.run_gemini_test` (the 3300-image test-set per-card Gemini pass, see
`docs/superpowers/plans/2026-08-07-confidence-calibration-experiments.md` and
CLAUDE.md's "Next steps" for context) on Kaggle instead of locally -- mainly
useful as a second, independent `GEMINI_API_KEY` quota source running in
parallel with a local invocation.

**Before running, attach four inputs to this notebook** (File -> Add Input):
1. The competition itself: `museumscat-specimen-collection-annotation-task`
   (raw images -- not actually needed by this script, only by the whole-image
   fallback runner, but wired up here too since it's cheap and this notebook
   may get reused for that later).
2. `museumscat-card-crops` (private dataset uploaded 2026-08-13, owner
   `matiaspoullain`) -- just the SAM-detection card crops
   (`data/processed/card_crops/`) now. Reused as-is across runs -- 877MB of
   crop images, so it only gets re-uploaded if the crops themselves change.
3. `museumscat-gemini-code` (private dataset, owner `matiaspoullain`, first
   created 2026-08-15) -- this project's `src/` code, since it isn't public
   on GitHub in a form this notebook can just `pip install`. Split out of
   `museumscat-card-crops` specifically so a code fix doesn't require
   re-uploading 877MB of crop images -- it's ~350KB, cheap to push as a new
   version (`kaggle datasets version -p <folder> -r zip -m "..."`) every
   time `src/` changes.
4. `museumscat-gemini-progress` (private dataset, owner `matiaspoullain`,
   first created 2026-08-14) -- just the two small progress files
   (`test_card_predictions.csv` + `test_relevance_flags.csv`), seeded from
   whatever transcription progress existed locally at upload time, so this
   run doesn't re-spend quota re-transcribing crops already done locally.
   Cheap to re-upload (~1MB) as a **new version of this same dataset**
   (`kaggle datasets version -p <folder> -m "..."`) each time local progress
   moves forward.

Both `museumscat-gemini-code` and `museumscat-gemini-progress` are kept
separate from `museumscat-card-crops` specifically so refreshing either one
never has to touch (or risk dropping) the crops: `kaggle datasets version`
replaces a dataset's *entire* contents with whatever folder you point it at,
so any file missing from that folder would be dropped from the new version.

**Also required:**
- Settings -> Internet: **On** (needed to call the Gemini API).
- Add-ons -> Secrets -> add a secret named `GEMINI_API_KEY` (a free-tier
  Google AI Studio key, *different* from whatever key is used locally --
  using the same key in two places just splits one quota in half instead of
  adding a second one).

**Not done by this notebook:** pushing itself to Kaggle (`kaggle kernels
push`) -- that's a separate, deliberate step once this has been reviewed.
Also note there's an older `museum-card-crops-train-test` dataset from the
classical-pipeline era (see `notebooks/kaggle_finetune_and_predict_danish.ipynb`)
that looks like it may cover similar ground -- worth checking whether it's
still needed alongside this new one.

## Cell 1: locate the four attached inputs

In [ ]:
import glob
import os


def _find_dataset_dir(slug):
    """Kaggle mounts attached datasets under /kaggle/input/<slug>/ almost
    always, but competition data sometimes nests under
    /kaggle/input/competitions/<slug>/ instead -- glob for it rather than
    hardcoding one path (same pattern as notebooks/kaggle_finetune_and_predict_danish.ipynb)."""
    candidates = glob.glob(f"/kaggle/input/**/{slug}", recursive=True)
    for c in candidates:
        if os.path.isdir(c):
            return c
    raise FileNotFoundError(f"dataset '{slug}' not found under /kaggle/input (checked: {candidates})")


COMPETITION_SLUG = "museumscat-specimen-collection-annotation-task"
CROPS_SLUG = "museumscat-card-crops"
CODE_SLUG = "museumscat-gemini-code"
PROGRESS_SLUG = "museumscat-gemini-progress"

COMPETITION_DIR = _find_dataset_dir(COMPETITION_SLUG)
CROPS_DIR = _find_dataset_dir(CROPS_SLUG)
CODE_DIR = _find_dataset_dir(CODE_SLUG)
PROGRESS_DIR = _find_dataset_dir(PROGRESS_SLUG)

print("competition data:", COMPETITION_DIR, os.listdir(COMPETITION_DIR))
print("card-crops dataset:", CROPS_DIR, os.listdir(CROPS_DIR))
print("code dataset:", CODE_DIR, os.listdir(CODE_DIR)[:5], "...")
print("progress dataset:", PROGRESS_DIR, os.listdir(PROGRESS_DIR))


## Cell 2: install the one dependency Kaggle's image doesn't already have

In [ ]:
!pip install -q google-genai


## Cell 3: GEMINI_API_KEY from Kaggle Secrets

Requires a secret named `GEMINI_API_KEY` (Add-ons -> Secrets in the notebook
editor) -- use a *different* free-tier key than the one used for local runs,
since the point of running this on Kaggle is a second independent quota
source, not splitting the same one.

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
print("GEMINI_API_KEY set:", bool(os.environ.get("GEMINI_API_KEY")))


## Cell 4: locate a (possibly still-zipped) entry inside the crops dataset

Kaggle auto-extracts uploaded `.zip` files into a same-named folder, but
handle the un-extracted case too rather than assume which one happened.
(`museumscat-gemini-code` and `museumscat-gemini-progress` are both small
and uploaded flat -- `CODE_DIR`/`PROGRESS_DIR` from Cell 1 are used directly
in Cells 5-6, no locating needed.)

In [ ]:
import shutil
import zipfile

_SCRATCH = "/kaggle/working/_crops_scratch"


def locate(name):
    """Absolute path to CROPS_DIR's `name` entry, extracting `name.zip` into
    _SCRATCH first if Kaggle left it zipped instead of auto-extracting it."""
    as_dir = os.path.join(CROPS_DIR, name)
    if os.path.isdir(as_dir):
        return as_dir
    as_zip = os.path.join(CROPS_DIR, f"{name}.zip")
    extracted = os.path.join(_SCRATCH, name)
    if not os.path.isdir(extracted):
        with zipfile.ZipFile(as_zip) as zf:
            zf.extractall(os.path.join(_SCRATCH, f"_{name}_unzip"))
        # the zip's own top-level entry is named `name` too (that's what got
        # zipped), so this is where it lands after extraction
        shutil.move(os.path.join(_SCRATCH, f"_{name}_unzip", name), extracted)
    return extracted


CARD_CROPS_SRC = locate("card_crops")
print(CARD_CROPS_SRC)


## Cell 5: unpack this project's code and cd into it

`museumscat-gemini-code` uploads a single `src.zip`, and Kaggle flattens a
dataset that consists of just one zip -- so `CODE_DIR` already contains the
`.py` files directly (no `src/` subfolder to strip).

In [ ]:
REPO_DIR = "/kaggle/working/repo"
os.makedirs(os.path.join(REPO_DIR, "src"), exist_ok=True)
for py_file in glob.glob(os.path.join(CODE_DIR, "*.py")):
    shutil.copyfile(py_file, os.path.join(REPO_DIR, "src", os.path.basename(py_file)))

os.chdir(REPO_DIR)
print(os.getcwd(), "->", sorted(os.listdir("src"))[:5], "...")


## Cell 6: wire up `data/` from the crops and progress inputs

Same relative paths `src/` expects locally (`data/raw/...`,
`data/processed/card_crops/...`, `data/processed/gemini/...`). Card-crop
*images* are symlinked (877MB, read-only, never written to); everything this
script might write to -- the relevance-flag cache and the output predictions
CSV -- gets a real writable copy seeded from the progress dataset instead,
since `/kaggle/input` is effectively read-only from this script's
perspective, and a symlink into it would make any write crash.

In [ ]:
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed/card_crops", exist_ok=True)
os.makedirs("data/processed/gemini", exist_ok=True)

# raw competition data (test.csv is what run_gemini_test.py actually reads;
# train.csv + images are here for completeness / future scripts, not this one)
shutil.copyfile(os.path.join(COMPETITION_DIR, "test.csv"), "data/raw/test.csv")
shutil.copyfile(os.path.join(COMPETITION_DIR, "train.csv"), "data/raw/train.csv")
if not os.path.exists("data/raw/images"):
    os.symlink(os.path.join(COMPETITION_DIR, "images"), "data/raw/images")

# card crops: manifest (tiny, real copy) + images (877MB, symlink -- read-only, fine)
shutil.copyfile(os.path.join(CARD_CROPS_SRC, "manifest.csv"), "data/processed/card_crops/manifest.csv")
if not os.path.exists("data/processed/card_crops/images"):
    os.symlink(os.path.join(CARD_CROPS_SRC, "images"), "data/processed/card_crops/images")

# seed progress: writable copies, not symlinks (see cell docstring above)
for fname in ("test_relevance_flags.csv", "test_card_predictions.csv"):
    src_path = os.path.join(PROGRESS_DIR, fname)
    if os.path.isfile(src_path):
        shutil.copyfile(src_path, os.path.join("data/processed/gemini", fname))

print("manifest rows:", sum(1 for _ in open("data/processed/card_crops/manifest.csv")) - 1)
print("crop images:", len(os.listdir("data/processed/card_crops/images")))
print("seeded predictions:",
      sum(1 for _ in open("data/processed/gemini/test_card_predictions.csv")) - 1
      if os.path.exists("data/processed/gemini/test_card_predictions.csv") else 0)


## Cell 7: sanity check before spending any quota

Reproduces `run_gemini_test.main()`'s setup (relevance flags + pending-crop
count) without calling the Gemini API, so a wiring mistake shows up here
instead of after burning real requests.

In [ ]:
import pandas as pd

from src.card_relevance_visual import TEST_RELEVANCE_CACHE_PATH, load_or_compute_relevance_flags
from src.gemini_batch_runner import already_done, pending_crops

OUTPUT_PATH = "data/processed/gemini/test_card_predictions.csv"

test_image_files = pd.read_csv("data/raw/test.csv")["image_file"].tolist()
test_specimens = {os.path.splitext(f)[0] for f in test_image_files}
print(f"{len(test_specimens)} test specimens")

relevance_flags = load_or_compute_relevance_flags(test_specimens, TEST_RELEVANCE_CACHE_PATH)
print(f"relevance flags ready for {len(relevance_flags)} crops")

pending = pending_crops(test_image_files, relevance_flags)
done = already_done(OUTPUT_PATH)
still_pending = [c for c in pending if c[0] not in done]
print(f"{len(pending)} relevant crops total, {len(done)} already done, {len(still_pending)} still pending")


## Cell 8: run the batch transcription

`--limit` caps how many *pending* crops this single invocation processes --
keep it well under Cell 7's `still_pending` count so the run ends cleanly
instead of hitting Kaggle's session time limit mid-write. Re-running this
cell (or a fresh session with the updated output re-attached, see Cell 9)
picks up where it left off automatically -- `run_batch` skips any
`crop_filename` already present in `OUTPUT_PATH`.

In [ ]:
LIMIT = 1500  # tune against Cell 7's still_pending count and remaining session time

!python -m src.run_gemini_test --rpm 10 --limit {LIMIT}


## Cell 9: persist progress for the next session

Kaggle resets `/kaggle/working` between sessions -- **Save Version** (or
"Save & Run All") before closing so this output survives as the notebook's
own output. To resume with a *fresh* seed (recommended, since it also picks
up anything the local run added meanwhile): download
`test_card_predictions.csv` from this run's output, merge it with the local
copy (concatenate + drop duplicate `crop_filename` rows, keeping either --
both should agree since they're the same crop, just possibly transcribed
twice by different keys), and upload the merged file as a **new version of
the small `museumscat-gemini-progress` dataset** (not `museumscat-card-crops`
-- that one stays untouched):

```
kaggle datasets version -p data/processed/gemini/kaggle_progress_dataset -m "..."
```

before the next Kaggle run (re-attaching the dataset in a new session picks
up the latest version automatically; an already-open session needs the input
re-added/refreshed).

In [ ]:
print("rows now:", sum(1 for _ in open(OUTPUT_PATH)) - 1)
print(f"output stays at {os.path.join(REPO_DIR, OUTPUT_PATH)} -- visible in the Output pane after Save Version")
